# ۲. مدل پایهٔ SURT و اجرای استریم

برای دانلود Internet کافی است؛ GPU فقط هنگام اجرای واقعی مدل لازم است. این نوت‌بوک به WhisperRT نیاز ندارد.
دو چیز متفاوت داریم: **وزن رسمیِ قابل دانلود** و **خروجی native streaming سازگار با پروژه**.
وزن رسمی به‌تنهایی فایل‌های `mask.pt`/`encoder.pt` با رابط `forward_chunk` را نمی‌سازد؛ مبدل/اتصال آن هنوز در پروژه پیاده‌سازی نشده است.
این کمبود با اجرای یک سلول یا فعال‌کردن GPU حل نمی‌شود. نوت‌بوک وضعیت را گزارش می‌کند و اجرای واقعی را تا فراهم‌شدن مدل سازگار متوقف می‌کند.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/manzourali/English-Streaming-ASR.git"
REPOSITORY_REF = "master"
ROOT = Path("/kaggle/working/English-Streaming-ASR")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(ROOT)], check=True)
os.chdir(ROOT)
if not (ROOT / "scripts/check_surt_assets.py").is_file():
    raise RuntimeError("This GitHub checkout predates the Kaggle workflow update. Publish the updated source, then start a fresh session.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[kaggle,dev]"], check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
ARTIFACTS = Path("/kaggle/working/asr_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

def run_script(name, *arguments):
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *map(str, arguments)], check=True)

In [ ]:
try:
    import torch
    gpu_available = torch.cuda.is_available()
    print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "GPU:", gpu_available)
except ImportError:
    gpu_available = False
    print("No PyTorch; asset download can still run on CPU.")
DATA_ROOT = Path("/kaggle/working/asr_artifacts/data/pilot_v1")
# In a fresh session, replace DATA_ROOT with the attached previous notebook output.
EXPORT_ROOT = None
MODEL_VARIANT = "surt_base"
DOWNLOAD_OFFICIAL_WEIGHTS = True
MANIFEST_SPLIT = "validation"
assert (DATA_ROOT / "manifests" / f"{MANIFEST_SPLIT}.jsonl").is_file(), "Attach step 1 output and set DATA_ROOT."

## دانلود مدل رسمی و ثبت نسخه

مخزن رسمیِ معرفی‌شده توسط icefall: `desh2608/icefall-surt-libricss-dprnn-zipformer`.
برای شروع `surt_base` انتخاب شده است. کد ابتدا SHA نسخه را می‌گیرد و همان نسخه را دانلود می‌کند.
فقط شاخهٔ مدل انتخابی و واژگان دانلود می‌شوند؛ کل مدل‌های base/large باهم دانلود نمی‌شوند.
این دانلود، آموزش یا ساخت native export نیست. `zipformer_base.pt` تنها بخش تشخیص اولیه است؛ جایگزین SURT کامل نیست.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
MODEL_REPOSITORY = "desh2608/icefall-surt-libricss-dprnn-zipformer"
if DOWNLOAD_OFFICIAL_WEIGHTS:
    revision = HfApi().model_info(MODEL_REPOSITORY).sha
    model_root = ARTIFACTS / "official_model" / MODEL_VARIANT
    snapshot_download(repo_id=MODEL_REPOSITORY, revision=revision, local_dir=model_root,
                      allow_patterns=[f"exp/{MODEL_VARIANT}/*", "data/lang_bpe_500/*", "README.md"])
    files = [str(path.relative_to(model_root)) for path in model_root.rglob("*.pt")]
    if not files:
        raise RuntimeError("The selected upstream model directory contains no checkpoint. Inspect the pinned repository tree.")
    (model_root / "download_provenance.json").write_text(json.dumps(
        {"repo_id": MODEL_REPOSITORY, "revision": revision, "variant": MODEL_VARIANT, "checkpoint_files": files}, indent=2))
    print("Downloaded official checkpoints:", files)

## بررسی مدل قابل‌اجرا

اگر native export ندارید، `EXPORT_ROOT=None` بماند؛ خروجی باید `native_export_missing` باشد.
نام فایل‌های checkpoint رسمی را عوض نکنید و آن‌ها را به‌عنوان export معرفی نکنید.
اگر متخصص پروژه export معتبر ساخته است، آن را با Add Input متصل و مسیر پوشه‌اش را اینجا وارد کنید.
اعتبارسنجی سکوت فقط اجرای رابط را می‌سنجد؛ صحت متن، انطباق ویژگی‌ها با checkpoint و استریم واقعی باید جدا آزمایش شوند.

In [ ]:
probe_command = [sys.executable, str(ROOT / "scripts/check_surt_assets.py")]
if EXPORT_ROOT is not None:
    if not gpu_available:
        raise RuntimeError("Enable a Kaggle GPU before probing native model inference.")
    EXPORT_ROOT = Path(EXPORT_ROOT)
    probe_command += ["--export-dir", str(EXPORT_ROOT), "--probe", "--device", "cuda"]
probe = subprocess.run(probe_command, check=True, capture_output=True, text=True)
readiness = json.loads(probe.stdout)
print(json.dumps(readiness, indent=2))
(ARTIFACTS / "surt_readiness.json").write_text(json.dumps(readiness, indent=2))

## اجرای یک نمونهٔ واقعی، سپس مجموعهٔ انتخابی

این سلول فقط با export قابل اجرا پیش می‌رود. ابتدا یک نمونه از validation؛ سپس با `MAX_EXAMPLES=None` مجموعهٔ انتخابی اجرا می‌شود.
تا پایان انتخاب تنظیمات، `test` را برای تنظیم مدل استفاده نکنید.
پارامتر `SYSTEM_NAME` برای خروجی وزن اولیه `surt_base` و برای وزن پس از آموزش `surt_finetuned` است؛ نام را مطابق وزن واقعی انتخاب کنید.

In [ ]:
import yaml
if not readiness.get("ready"):
    raise RuntimeError("Native SURT integration is still missing or failed. Downloaded weights alone cannot run this pipeline; see docs/KAGGLE_WORKFLOW_FA.md.")
SYSTEM_NAME = "surt_base"
MAX_EXAMPLES = 1
results_root = ARTIFACTS / "results" / SYSTEM_NAME
inference_override = ARTIFACTS / f"{SYSTEM_NAME}_inference.yaml"
inference_override.write_text(yaml.safe_dump({
    "experiment": {"system": SYSTEM_NAME},
    "data": {"manifest": str(DATA_ROOT / "manifests" / f"{MANIFEST_SPLIT}.jsonl"),
             "dataset_id": DATA_ROOT.name},
    "model": {"backend_factory": "streaming_asr.models.torch_surt:load_torchscript_backend",
              "checkpoint": str(EXPORT_ROOT), "device": "cuda"},
    "paths": {"outputs": str(results_root)}
}, sort_keys=False))
arguments = ["--override", inference_override]
if MAX_EXAMPLES is not None:
    arguments += ["--max-samples", MAX_EXAMPLES]
run_script("run_multitalker.py", *arguments)
print("Evaluation-ready files:", list((results_root / "predictions").glob("*_evaluation_records.jsonl")))

## خروجی و ادامه

خروجی واقعی: متن دو کانال، PI-WER با شمارش خطا/کلمات، زمان پردازش و فایل `*_evaluation_records.jsonl`.
نتیجهٔ یک نمونه برای تأیید اجرای کد است؛ نتیجهٔ پژوهشی باید روی مجموعهٔ ثابت بزرگ‌تر گزارش شود.
خروجی را ذخیره کنید. اگر فقط ASR با وزن آماده می‌خواهید، آموزش لازم نیست؛ مرحلهٔ 14 را انجام دهید.
برای پژوهشِ بهبود مدل، مرحلهٔ 13 اختیاری است و به اتصال آموزش جداگانه نیاز دارد.

مرجع: https://github.com/k2-fsa/icefall/tree/master/egs/libricss/SURT